# 07 · Channel coding

A forward error correction (FEC) code adds redundancy so that the receiver
can remove the errors the channel left.  This notebook covers:

1. codes as objects, `encode` and `decode`, and pre- versus post-FEC metrics;
2. hard and soft decoding of a block code (Hamming, Chase);
3. algebraic codes: Reed-Solomon and the bounded-distance threshold;
4. LDPC codes and iterative decoding: the waterfall;
5. **NGMI**, the metric that predicts whether a soft-decision code decodes;
6. polar, convolutional and turbo codes in one comparison.

It builds on the LLRs and the GMI of
[06 · AWGN and performance metrics](06-awgn-and-performance-metrics.ipynb).

In [ ]:
import time
from math import comb

import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device
from commkit.coding import (
    BCJR,
    SCL,
    Chase,
    ConvolutionalCode,
    LDPCCode,
    LinearBlockCode,
    MaximumLikelihood,
    MinSum,
    PolarCode,
    ReedSolomonCode,
    SumProduct,
    Syndrome,
    TurboCode,
    TurboIterations,
    decode,
    encode,
)

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
QPSK = Constellation.qam(4)


def host(a):
    return to_device(a, "cpu")


def coded_link(code, esn0_db, num_codewords, constellation=QPSK, seed=0):
    """A coded payload over AWGN at one sample per symbol, and its LLRs."""
    k = constellation.bits_per_symbol
    tx = ck.generate(
        constellation,
        num_codewords * code.n // k,
        symbol_rate=1e9,
        code=code,
        rng=seed,
        device=DEVICE,
    )
    rx = ck.impairments.apply_awgn(tx, esn0_db=esn0_db, rng=seed + 1)
    return rx, ck.mapping.compute_llr(rx, noise_var=10 ** (-esn0_db / 10))

## 1. Codes as objects

A code is a frozen value object with `n` transmitted and `k` information
bits per block.  `encode` maps information bits to codewords and `decode`
maps LLRs (or hard bits) back, returning a `DecodeResult` with the
decoded bits, the decoder's own success check (`valid`) and the iterations
used.  The decoder is chosen by a method object, as everywhere in CommKit.

In [ ]:
h74 = LinearBlockCode.hamming(3)
print(h74, "rate", h74.rate)
u = np.array([1, 0, 1, 1], dtype=np.int8)
c = encode(u, code=h74)
print("codeword:", c)
for pos in range(7):  # every single error is corrected
    e = c.copy()
    e[pos] ^= 1
    assert np.array_equal(decode(e, code=h74).bits, u)
print("all 7 single errors corrected")

`generate(..., code=)` builds a coded payload.  Its reference keeps two
bit streams: `bits`, the labels on the air, and `info_bits`, the source
bits.  Pre-FEC metrics compare the hard decisions with `bits`; post-FEC
metrics compare the decoder's output with `info_bits`.  For an uncoded
payload `info_bits` *is* `bits`, so the same calls work unchanged.

In [ ]:
rx, llrs = coded_link(h74, esn0_db=9.0, num_codewords=20000)
ref = rx.reference
print("code block (n, k):", ref.code_block)
res = decode(llrs, code=h74)
print(f"pre-FEC BER  {ck.metrics.ber(rx):.2e}")
print(f"post-FEC BER {ck.metrics.ber(res.bits, ref.info_bits):.2e}")
print(f"post-FEC FER {ck.metrics.fer(res.bits, ref.info_bits, block_length=4):.2e}")

## 2. Hard and soft decoding

Syndrome decoding uses only the hard decisions.  A soft decoder uses the
LLR magnitudes too.  Maximum-likelihood decoding tries all `2^k` codewords,
which is affordable only for small codes.  **Chase** decoding flips the
least reliable bits in all combinations and hard-decodes each test word.
It approaches ML at a fraction of the cost, and it is how the
Hamming(128,119) inner code of 400ZR is decoded in
[21 · FEC in optical standards](21-fec-in-optical-standards.ipynb).

In [ ]:
h16 = LinearBlockCode.hamming(4, extended=True)  # (16, 11), distance 4
methods = {
    "syndrome (hard)": Syndrome(),
    "Chase, 3 positions": Chase(3, hard=Syndrome(max_weight=1)),
    "maximum likelihood": MaximumLikelihood(),
}
grid = np.arange(4.0, 10.1, 1.0)
post = {name: [] for name in methods}
pre = []
for e in grid:
    rx, llrs = coded_link(h16, e, num_codewords=5000, seed=int(e))
    pre.append(ck.metrics.ber(rx))
    for name, m in methods.items():
        res = decode(llrs, code=h16, method=m)
        post[name].append(ck.metrics.ber(res.bits, rx.reference.info_bits))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy(grid, pre, "k--", label="pre-FEC")
for name, values in post.items():
    ax.semilogy(grid, values, "o-", label=name)
ax.set_xlabel("Es/N0 [dB] (QPSK)")
ax.set_ylabel("BER")
ax.legend()
plt.show()
print("post-FEC BER at 7 dB:", {k: f"{v[3]:.1e}" for k, v in post.items()})

At 7 dB soft decoding lowers the post-FEC BER more than tenfold.  Chase with three test positions matches
maximum-likelihood decoding on this code at a fraction of the cost.

## 3. Algebraic codes: Reed-Solomon

A Reed-Solomon code over GF(2^m) corrects any `t = (n - k) / 2` wrong
*symbols* per codeword, whatever the bit errors inside them, and nothing
beyond: decoding is bounded-distance.  So the frame error rate is the
probability of more than `t` symbol errors, a binomial tail.  The KP4 code
RS(544, 514) of 200G-800G Ethernet has `t = 15` 10-bit symbols.  Its
threshold, a pre-FEC BER of 2.4e-4, leaves a frame error rate around 1e-12,
which is too rare to simulate but follows from the tail.

In [ ]:
kp4 = ReedSolomonCode.kp4()
print(kp4, "t =", kp4.t)


def kp4_fer(p_bit):
    p_sym = 1 - (1 - p_bit) ** 10
    return sum(
        comb(544, i) * p_sym**i * (1 - p_sym) ** (544 - i) for i in range(16, 545)
    )


rng = np.random.default_rng(0)
u = rng.integers(0, 2, (400, kp4.k), dtype=np.int8)
words = host(encode(u, code=kp4))
for p in (3e-3, 4e-3, 5e-3):
    flipped = words ^ (rng.random(words.shape) < p)
    res = decode(ck.backend.to_device(flipped, DEVICE), code=kp4)
    fer = float(np.mean(np.any(host(res.bits) != u, axis=1)))
    print(
        f"pre-FEC BER {p:.0e}: simulated FER {fer:.3f}, binomial tail {kp4_fer(p):.3f}"
    )
print(f"at the 2.4e-4 threshold the tail is {kp4_fer(2.4e-4):.1e}")

## 4. LDPC codes and iterative decoding

LDPC codes are decoded by passing messages along a sparse graph of parity
checks.  They decode almost error-free above a sharp threshold (the
*waterfall*), close to the GMI.  `LDPCCode.dvbs2` gives the DVB-S2 codes,
the common stand-in for the proprietary soft-decision FEC of coherent
modems.  `MinSum` (normalized min-sum) is the hardware decoder and
`SumProduct` exact belief propagation.  Both run layered, on the GPU as
one CUDA kernel.

In [ ]:
ldpc = LDPCCode.dvbs2("1/2", "short")
print(ldpc, f"rate {ldpc.rate:.3f}")
grid = np.arange(-0.5, 1.01, 0.25)
fer = {"min-sum": [], "sum-product": []}
ngmi = []
for e in grid:
    rx, llrs = coded_link(ldpc, e, num_codewords=24, seed=int(100 + 10 * e))
    ngmi.append(ck.metrics.ngmi(rx, noise_var=10 ** (-e / 10)))
    for name, m in (("min-sum", MinSum()), ("sum-product", SumProduct())):
        res = decode(llrs, code=ldpc, method=m)
        fer[name].append(
            ck.metrics.fer(res.bits, rx.reference.info_bits, block_length=ldpc.k)
        )
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, values in fer.items():
    ax.semilogy(grid, np.maximum(values, 1e-3), "o-", label=name)
ax.set_xlabel("Es/N0 [dB] (QPSK)")
ax.set_ylabel("FER (floor at 1e-3: no errors)")
ax.legend()
plt.show()
for e, a, b in zip(grid, fer["min-sum"], fer["sum-product"], strict=True):
    print(f"{e:4.2f} dB: FER min-sum {a:.3f}, sum-product {b:.3f}")

## 5. NGMI: will the code decode?

The normalized GMI, `NGMI = 1 - (H(X) - GMI) / m`, is 1 for perfect LLRs.
A soft-decision code of rate `R` decodes once the NGMI exceeds a threshold
somewhat above `R`, and that threshold hardly depends on the modulation.
This is why optical systems quote FEC thresholds as NGMI.  For this
rate-1/2 code:

In [ ]:
for e, n, f in zip(grid, ngmi, fer["sum-product"], strict=True):
    print(f"Es/N0 {e:4.2f} dB: NGMI {n:.3f}  FER {f:.3f}")

The frames start decoding once the NGMI passes about 0.5, roughly 0.06
above the code rate (0.444).  The gap measures how far this code is from
capacity at its length of 16200 bits.  Sum-product decodes slightly
earlier than min-sum, about 0.1 dB, and costs more per iteration.

## 6. Polar, convolutional and turbo codes

Three more families, each with its standard decoder: 5G NR polar codes
with CRC-aided list decoding, the classic `K = 7` convolutional code with
Viterbi decoding, and the LTE turbo code with iterative BCJR decoding.
All have rate about 1/2 or 1/3 and a few hundred information bits:

In [ ]:
codes = {
    "polar (256, 128) + CRC-11, SCL(8)": (PolarCode.nr(128, 256, crc="11"), SCL(8)),
    "convolutional K=7, k=128, Viterbi": (ConvolutionalCode.k7(128), None),
    "convolutional K=7, k=128, BCJR": (ConvolutionalCode.k7(128), BCJR()),
    "turbo LTE K=128, 8 iterations": (TurboCode.lte(128), TurboIterations()),
}
for name, (code, method) in codes.items():
    ebn0 = 2.5
    esn0 = ebn0 + 10 * np.log10(2 * code.rate)
    start = time.perf_counter()
    rx, llrs = coded_link(code, esn0, num_codewords=300, seed=7)
    res = decode(llrs, code=code, method=method)
    f = ck.metrics.fer(res.bits, rx.reference.info_bits, block_length=code.k)
    print(f"{name:38s} rate {code.rate:.3f}  FER at Eb/N0 2.5 dB: {f:.3f}")

At the same energy per information bit, the turbo code (rate 1/3) and the
polar list decoder lose the fewest frames, and the 64-state convolutional
code the most.  BCJR and Viterbi make the same decisions here; BCJR also
returns the soft output that iterative receivers need.

**Summary**

* A code is an object; `encode` and `decode` work on bits and LLRs, and
  `decode` takes a method object for the decoder.
* `generate(..., code=)` keeps both the bits on the air and the source
  bits, so pre-FEC (`ber(rx)`) and post-FEC (`ber(res.bits, info_bits)`,
  `fer`) metrics are one call each.
* Hard decoding throws away the reliabilities; soft decoding (Chase, ML,
  belief propagation) uses them.
* Bounded-distance RS codes have a binomial-tail threshold; LDPC codes a
  waterfall predicted by the NGMI.

**Next:** [08 · Probabilistic shaping and coded modulation](08-probabilistic-shaping-and-coded-modulation.ipynb).